In [1]:
import numpy as np
import pandas as pd
from io import StringIO
from unlzw3 import unlzw
from scipy.io import loadmat
from scipy.sparse import lil_matrix, coo_matrix
from sklearn.model_selection import train_test_split

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.3

# AAC and Dipeptide Protein Dataset

In [3]:
train_data = pd.read_csv("../Datasets/AAC and Dipeptide Protein Dataset/AFP300_nonAFP300_train_AACandDipeptide_twoSeg.csv")
train_data.rename(columns={1: "label"}, inplace=True)
train_data.columns = [["label"] + list(range(len(train_data.columns))[:-1])]
train_data = train_data[list(range(len(train_data.columns))[:-1]) + ["label"]].copy()
train_data.columns = [col[0] if isinstance(col, tuple) else col for col in train_data.columns]
train_data["subset"] = "train"
train_data

,0,1,2,3,4,5,6,7,8,9,...,832,833,834,835,836,837,838,839,label,subset
0,4.444,6.667,2.963,6.667,3.704,3.704,4.444,9.630,5.185,9.630,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0,train
1,10.405,0.578,6.358,7.514,2.890,7.514,2.312,2.312,1.156,9.249,...,0.581,0.000,0.000,0.581,0.581,0.000,0.000,0.000,0,train
2,3.448,2.299,5.747,7.471,4.023,4.023,1.724,7.471,8.046,16.667,...,0.000,0.575,0.000,0.575,0.575,0.575,0.575,1.149,0,train
3,8.876,1.775,3.550,7.101,2.367,8.284,5.325,3.550,2.959,9.467,...,0.000,0.000,0.000,0.000,1.190,0.595,0.595,0.000,0,train
4,9.694,1.020,2.551,17.857,0.510,5.612,2.551,2.041,3.571,8.163,...,0.000,0.000,0.000,0.510,0.000,0.000,0.000,0.000,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,4.795,5.479,3.425,4.795,2.055,4.795,2.055,4.795,4.110,10.959,...,0.000,0.000,0.000,0.000,0.000,0.685,0.685,0.685,1,train
596,3.614,0.000,6.627,8.434,3.614,4.819,1.807,4.217,6.627,16.265,...,0.000,0.000,0.000,0.000,0.000,0.000,0.606,0.000,1,train
597,3.478,6.957,2.609,6.087,4.348,6.957,0.000,4.348,11.304,9.565,...,0.000,0.000,0.000,0.870,0.000,0.000,0.870,0.870,1,train
598,9.375,3.906,2.344,7.031,2.344,3.125,2.344,0.781,2.344,14.063,...,0.000,0.781,0.000,0.000,0.000,0.781,0.000,0.781,1,train


In [4]:
test_data = pd.read_csv("../Datasets/AAC and Dipeptide Protein Dataset/AFP181_nonAFP9193_test_AACandDipeptide_twoSeg.csv")
test_data.rename(columns={1: "label"}, inplace=True)
test_data.columns = [["label"] + list(range(len(test_data.columns))[:-1])]
test_data = test_data[list(range(len(test_data.columns))[:-1]) + ["label"]].copy()
test_data.columns = [col[0] if isinstance(col, tuple) else col for col in test_data.columns]
test_data["subset"] = "test"
test_data

,0,1,2,3,4,5,6,7,8,9,...,832,833,834,835,836,837,838,839,label,subset
0,3.333,3.333,3.333,10.000,0.000,10.000,0.000,3.333,6.667,6.667,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0,test
1,11.429,0.000,4.286,15.714,5.714,1.429,2.857,1.429,4.286,11.429,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0,test
2,4.255,0.000,4.255,6.383,8.511,2.128,0.000,19.149,8.511,6.383,...,0.000,0.000,0.000,2.174,0.000,0.000,0.000,0.000,0,test
3,3.153,5.405,4.054,8.108,4.955,1.802,2.703,9.459,9.009,10.360,...,0.000,0.450,0.450,0.000,0.450,0.000,0.000,0.000,0,test
4,2.778,0.000,16.667,22.222,8.333,0.000,0.000,8.333,8.333,2.778,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9369,11.938,1.860,5.736,5.426,2.016,9.147,2.016,2.171,2.016,9.612,...,0.155,0.155,0.155,0.311,0.155,0.000,0.000,0.000,1,test
9370,8.974,1.282,1.282,3.846,1.282,5.128,1.282,6.410,3.846,14.103,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,test
9371,8.621,1.149,5.747,7.471,4.023,4.023,2.874,9.770,6.322,8.046,...,0.575,0.000,0.000,0.000,0.000,0.575,0.000,0.575,1,test
9372,6.422,3.670,4.587,11.009,2.752,3.670,1.835,6.422,16.514,10.092,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,test


In [5]:
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,832,833,834,835,836,837,838,839,label,subset
0,4.444,6.667,2.963,6.667,3.704,3.704,4.444,9.630,5.185,9.630,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0,train
1,10.405,0.578,6.358,7.514,2.890,7.514,2.312,2.312,1.156,9.249,...,0.581,0.000,0.000,0.581,0.581,0.000,0.000,0.000,0,train
2,3.448,2.299,5.747,7.471,4.023,4.023,1.724,7.471,8.046,16.667,...,0.000,0.575,0.000,0.575,0.575,0.575,0.575,1.149,0,train
3,8.876,1.775,3.550,7.101,2.367,8.284,5.325,3.550,2.959,9.467,...,0.000,0.000,0.000,0.000,1.190,0.595,0.595,0.000,0,train
4,9.694,1.020,2.551,17.857,0.510,5.612,2.551,2.041,3.571,8.163,...,0.000,0.000,0.000,0.510,0.000,0.000,0.000,0.000,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9969,11.938,1.860,5.736,5.426,2.016,9.147,2.016,2.171,2.016,9.612,...,0.155,0.155,0.155,0.311,0.155,0.000,0.000,0.000,1,test
9970,8.974,1.282,1.282,3.846,1.282,5.128,1.282,6.410,3.846,14.103,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,test
9971,8.621,1.149,5.747,7.471,4.023,4.023,2.874,9.770,6.322,8.046,...,0.575,0.000,0.000,0.000,0.000,0.575,0.000,0.575,1,test
9972,6.422,3.670,4.587,11.009,2.752,3.670,1.835,6.422,16.514,10.092,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1,test


In [6]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 0    0.951775
 1    0.048225
 Name: proportion, dtype: float64,
 subset
 test     0.939844
 train    0.060156
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.980691
         1        0.019309
 train   0        0.500000
         1        0.500000
 Name: proportion, dtype: float64,
 np.int64(0))

In [7]:
data.to_parquet("../Benchmark/aac_dipeptide_protein.parquet", index=False)

# CS 4375 Term Project - Classification

In [8]:
train_data = pd.read_csv("../Datasets/CS 4375 Term Project - Classification/train.csv")
train_data.rename(columns={"Class": "label"}, inplace=True)
train_data.columns = [["label"] + list(range(len(train_data.columns))[:-1])]
train_data = train_data[list(range(len(train_data.columns))[:-1]) + ["label"]].copy()
train_data.columns = [col[0] if isinstance(col, tuple) else col for col in train_data.columns]
train_data["subset"] = "train"
train_data

,0,1,2,3,4,5,6,7,8,9,...,168,169,170,171,172,173,174,175,label,subset
0,0,1,1.260,1.17,0.720,4.59,0.45,0.765,0.540,0.495,...,2,1,3,0,0,0,0,1,0,train
1,1,0,0.450,0.81,0.000,0.00,0.00,0.855,0.000,1.170,...,1,1,3,0,0,0,0,1,0,train
2,0,1,0.540,2.88,0.000,0.00,0.00,0.765,0.000,0.000,...,2,1,3,0,0,0,0,1,0,train
3,0,1,0.810,1.35,0.450,0.00,0.00,0.000,0.720,0.900,...,2,1,3,0,0,0,0,1,0,train
4,0,1,0.900,1.17,0.765,0.00,0.00,0.630,0.810,0.000,...,2,1,3,0,0,0,0,1,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,0,0,0.990,230.85,0.630,0.00,0.00,0.810,0.000,0.000,...,2,1,3,0,0,0,0,0,1,train
15996,0,0,0.765,1.71,0.810,0.00,0.00,0.810,0.000,0.000,...,1,1,3,0,0,0,0,1,1,train
15997,0,1,11.070,1.08,0.810,0.00,0.00,0.810,0.720,0.000,...,2,1,3,0,0,0,0,1,0,train
15998,1,0,0.450,1.26,0.540,1.08,0.00,0.630,0.585,0.810,...,2,1,3,0,0,0,0,1,0,train


In [9]:
test_data = pd.read_csv("../Datasets/CS 4375 Term Project - Classification/prefinal-test.csv")
test_data.rename(columns={"Class": "label"}, inplace=True)
test_data.columns = [["label"] + list(range(len(test_data.columns))[:-1])]
test_data = test_data[list(range(len(test_data.columns))[:-1]) + ["label"]].copy()
test_data.columns = [col[0] if isinstance(col, tuple) else col for col in test_data.columns]
test_data["subset"] = "test"
test_data

,0,1,2,3,4,5,6,7,8,9,...,168,169,170,171,172,173,174,175,label,subset
0,0,1,0.45,1.08,0.000,0.000,0.00,1.080,0.720,0.495,...,1,1,3,0,0,0,0,1,0,test
1,0,0,0.81,1.08,0.630,0.900,0.99,0.720,0.720,0.990,...,2,1,4,0,0,0,0,0,1,test
2,1,0,1.62,2.43,0.000,0.000,0.00,1.530,0.720,0.000,...,1,1,3,0,0,0,0,0,0,test
3,0,0,0.27,6.75,0.000,0.000,0.00,1.125,0.000,0.000,...,1,1,2,0,0,0,0,0,1,test
4,0,0,0.54,1.26,0.810,0.810,0.00,0.675,0.450,1.035,...,2,1,3,0,0,0,0,0,1,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,0,1,0.00,146.70,0.000,0.000,0.00,0.630,0.000,0.000,...,2,1,3,0,0,0,0,1,0,test
3996,0,0,1.62,1.17,0.810,0.000,0.00,0.675,0.000,0.000,...,1,1,2,0,0,0,0,0,1,test
3997,1,0,0.63,89.91,0.000,0.000,0.00,0.000,0.000,0.900,...,2,1,3,2,0,0,0,0,0,test
3998,0,1,0.99,1.35,0.855,0.585,0.99,0.810,0.000,0.000,...,1,1,3,0,0,0,0,0,0,test


In [10]:
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data

,0,1,2,3,4,5,6,7,8,9,...,168,169,170,171,172,173,174,175,label,subset
0,0,1,1.26,1.17,0.720,4.590,0.45,0.765,0.540,0.495,...,2,1,3,0,0,0,0,1,0,train
1,1,0,0.45,0.81,0.000,0.000,0.00,0.855,0.000,1.170,...,1,1,3,0,0,0,0,1,0,train
2,0,1,0.54,2.88,0.000,0.000,0.00,0.765,0.000,0.000,...,2,1,3,0,0,0,0,1,0,train
3,0,1,0.81,1.35,0.450,0.000,0.00,0.000,0.720,0.900,...,2,1,3,0,0,0,0,1,0,train
4,0,1,0.90,1.17,0.765,0.000,0.00,0.630,0.810,0.000,...,2,1,3,0,0,0,0,1,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0,1,0.00,146.70,0.000,0.000,0.00,0.630,0.000,0.000,...,2,1,3,0,0,0,0,1,0,test
19996,0,0,1.62,1.17,0.810,0.000,0.00,0.675,0.000,0.000,...,1,1,2,0,0,0,0,0,1,test
19997,1,0,0.63,89.91,0.000,0.000,0.00,0.000,0.000,0.900,...,2,1,3,2,0,0,0,0,0,test
19998,0,1,0.99,1.35,0.855,0.585,0.99,0.810,0.000,0.000,...,1,1,3,0,0,0,0,0,0,test


In [11]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64'), dtype('float64')], dtype=object),
 label
 0    0.6649
 1    0.3351
 Name: proportion, dtype: float64,
 subset
 train    0.8
 test     0.2
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.671250
         1        0.328750
 train   0        0.663312
         1        0.336687
 Name: proportion, dtype: float64,
 np.int64(0))

In [12]:
data.to_parquet("../Benchmark/cs4375_term_project.parquet", index=False)

# 21st Century Bordeaux data Dataset

In [ ]:
data = pd.read_csv("../Datasets/21st Century Bordeaux Wine Dataset/BordeauxWines.csv")
merge = data.iloc[:, :4]
data = data.iloc[:, 4:]

In [ ]:
for col in data.select_dtypes(include=["float", "float64"]).columns:
    data[col] = data[col].astype("category")

cols_to_keep = [col for col in data.columns if not (data[col].dtype.name == "category" and len(data[col].cat.categories) == 1)]
data = data[cols_to_keep].copy()

data["label"] = np.arange(1, len(data) + 1)

data = pd.concat([data, merge.reset_index(drop=True)], axis=1)

data["label"] = np.where(data["Score"] >= 90, 1, 0)
data["label"] = data["label"].astype("category")

front_cols = ["label", "Score", "Name", "Year", "Price"]
other_cols = [c for c in data.columns if c not in front_cols]
data = data[front_cols + other_cols]

data = data.drop(columns=["Year", "Name", "Price", "Score"])
data

,label,BLOOD ORANGE,CITRUS,CITRUS PEEL,CITRUS ZEST,CLEMENTINE,LIME,GRAPEFRUIT,GRAPEFRUIT PEEL,ORANGE,...,SKUNK,SULFUR DIOXIDE,"WET WOOL,WET DOG",ACETIC ACID,ETHANOL,ETHYL ACETATE,ALCOHOL,FROTH,MENTHOL,SHERRY
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14344,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14345,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14346,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
14347,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
data.columns = [["label"] + list(range(len(data.columns))[:-1])]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]

In [15]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [16]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,977,978,979,980,981,982,983,984,label,subset
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,train
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14344,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
14345,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
14346,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test
14347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,test


In [17]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('int64')], dtype=object),
 label
 0    0.702906
 1    0.297094
 Name: proportion, dtype: float64,
 subset
 train    0.699979
 test     0.300021
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.702904
         1        0.297096
 train   0        0.702907
         1        0.297093
 Name: proportion, dtype: float64,
 np.int64(0))

In [18]:
data.to_parquet("../Benchmark/bordeaux_wine.parquet", index=False)

# High Dimensional Datascape

In [9]:
data = pd.read_csv("../Datasets/High Dimensional Datascape/all_data.csv")
print(data.shape)
display(data.head(3))

(230, 537)


,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 527,Unnamed: 528,Unnamed: 529,Unnamed: 530,Unnamed: 531,Unnamed: 532,Unnamed: 533,Unnamed: 534,Unnamed: 535,Label
0,-0.000133,0.000262,0.001099,0.001834,0.002109,0.002223,0.002233,0.002036,0.001582,0.000969,...,0.82953,2.9079,3.7557,1.3344,0.74247,0.22507,0.56249,1.5705,0.79906,0
1,-0.000842,-0.001011,-0.001071,-0.000944,-0.000794,-0.000610,-0.000445,-0.000173,0.000077,0.000285,...,0.84335,3.0110,3.9877,1.2461,0.74423,0.22567,0.61034,1.6645,0.74574,0
2,-0.000766,-0.000535,0.000162,0.000898,0.001287,0.001582,0.001704,0.001659,0.001574,0.001438,...,0.87413,3.0613,3.9749,1.1560,0.52508,0.19934,0.45707,1.3386,0.74574,0


In [10]:
data.rename(columns={"Label": "label"}, inplace=True)
data.columns = [list(range(len(data.columns))[:-1]) + ["label"]]
data = data[list(range(len(data.columns))[:-1]) + ["label"]].copy()
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data.head(3)

,0,1,2,3,4,5,6,7,8,9,...,527,528,529,530,531,532,533,534,535,label
0,-0.000133,0.000262,0.001099,0.001834,0.002109,0.002223,0.002233,0.002036,0.001582,0.000969,...,0.82953,2.9079,3.7557,1.3344,0.74247,0.22507,0.56249,1.5705,0.79906,0
1,-0.000842,-0.001011,-0.001071,-0.000944,-0.000794,-0.000610,-0.000445,-0.000173,0.000077,0.000285,...,0.84335,3.0110,3.9877,1.2461,0.74423,0.22567,0.61034,1.6645,0.74574,0
2,-0.000766,-0.000535,0.000162,0.000898,0.001287,0.001582,0.001704,0.001659,0.001574,0.001438,...,0.87413,3.0613,3.9749,1.1560,0.52508,0.19934,0.45707,1.3386,0.74574,0


In [11]:
train_data, test_data = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=data["label"])

In [12]:
train_data["subset"] = "train"
test_data["subset"] = "test"
data = pd.concat([train_data, test_data], axis=0).reset_index(drop=True)
data.columns = [col[0] if isinstance(col, tuple) else col for col in data.columns]
data["label"] = data["label"].astype(int)
data

,0,1,2,3,4,5,6,7,8,9,...,528,529,530,531,532,533,534,535,label,subset
0,-0.001349,-0.000190,0.002482,0.004849,0.005775,0.006323,0.006716,0.007131,0.007397,0.007322,...,2.7215,3.7279,1.3013,1.53790,0.71874,0.84606,2.4844,2.6847,1,train
1,-0.000890,-0.001530,-0.002412,-0.002752,-0.002583,-0.002281,-0.002045,-0.001764,-0.001359,-0.000745,...,2.7163,3.6046,1.1870,1.02580,0.35099,0.57239,2.0742,1.8550,0,train
2,-0.000715,-0.001360,-0.002363,-0.002955,-0.003058,-0.003027,-0.002889,-0.002662,-0.002343,-0.001681,...,3.2446,3.8950,1.1523,0.63946,0.20828,0.47358,1.3386,1.5340,0,train
3,-0.001137,-0.000214,0.002016,0.004294,0.005748,0.007095,0.008366,0.009507,0.010325,0.010584,...,3.0516,3.4048,1.0841,1.33580,0.72243,0.68803,1.9513,2.4617,1,train
4,-0.001322,-0.002223,-0.003461,-0.003902,-0.003582,-0.003017,-0.002494,-0.002080,-0.001634,-0.000927,...,2.9385,3.6325,1.2562,1.40460,0.49462,0.72972,2.3192,1.9845,0,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,-0.001334,-0.000968,0.000148,0.001241,0.001653,0.001943,0.002151,0.002380,0.002652,0.002901,...,2.6365,2.9519,1.3715,1.57050,0.81209,0.81325,2.5649,2.7380,0,test
226,-0.001733,-0.001470,-0.000425,0.000672,0.001227,0.001940,0.002507,0.002894,0.003440,0.004160,...,2.7901,3.7207,1.4107,1.65810,0.78287,0.84145,2.8446,2.7913,1,test
227,0.000004,-0.001074,-0.003168,-0.004882,-0.005684,-0.006172,-0.006412,-0.006518,-0.006383,-0.005771,...,2.0589,3.6246,1.3955,1.56130,0.83451,0.87630,2.7380,2.7913,1,test
228,-0.000108,-0.000428,-0.001004,-0.001427,-0.001594,-0.001711,-0.001859,-0.001995,-0.001993,-0.001886,...,2.7057,3.3065,1.1585,1.24170,0.51356,0.73283,2.2641,1.8829,0,test


In [13]:
data.drop(columns=["subset"]).dtypes.unique(), data["label"].value_counts(normalize=True), data["subset"].value_counts(normalize=True), data.groupby("subset")["label"].value_counts(normalize=True), data.isna().sum().sum()

(array([dtype('float64'), dtype('int64')], dtype=object),
 label
 1    0.5
 0    0.5
 Name: proportion, dtype: float64,
 subset
 train    0.7
 test     0.3
 Name: proportion, dtype: float64,
 subset  label
 test    0        0.507246
         1        0.492754
 train   1        0.503106
         0        0.496894
 Name: proportion, dtype: float64,
 np.int64(0))

In [14]:
data.to_parquet("../Benchmark/high_dimensional_datascape.parquet", index=False)